# Détection d'anomalies routières — Étape 1 : Exploration du dataset

**Projet :** Deep Learning appliqué à la détection d'anomalies routières (dataset *RoadAnomaly*)
**Notebook :** `01_exploration_dataset.ipynb`

Avant de construire un modèle, il faut comprendre les données. Toutes les décisions des étapes
suivantes (résolution d'entrée, découpage train/validation/test, fonction de perte, métriques,
choix des architectures) s'appuient sur ce que ce notebook met en évidence.

**Questions traitées**

1. Comment le dataset est-il organisé et quelles annotations sont disponibles ?
2. Les données sont-elles propres et cohérentes (tailles, valeurs des masques, fichiers manquants) ?
3. Quels types d'anomalies existe-t-il, et combien d'exemples de chaque type ?
4. Quelle est la taille des anomalies, combien par image, et où se situent-elles dans l'image ?
5. Une règle simple (couleur, luminosité) suffit-elle à détecter les anomalies ?

| § | Section |
|---|---|
| 0 | Chargement des données dans Colab |
| 1 | Structure du dataset et annotations disponibles |
| 2 | Contrôles d'intégrité et métadonnées par image |
| 3 | Répartition des catégories |
| 4 | Inspection visuelle (+ images issues de la même source) |
| 5 | Formats d'annotation : masque sémantique vs instances |
| 6 | Déséquilibre des classes au niveau pixel |
| 7 | Analyse au niveau objet : nombre et taille |
| 8 | Répartition spatiale des anomalies |
| 9 | Apparence : la couleur suffit-elle ? |
| 10 | Synthèse et conséquences pour la suite |

## 0. Chargement des données dans Colab

**Avant d'exécuter :** sur votre ordinateur, faites un clic droit sur le dossier `RoadAnomaly`
→ *Envoyer vers → Dossier compressé* (ou *Compresser en fichier ZIP*). Vous obtenez
`RoadAnomaly.zip`.

Exécutez la cellule ci-dessous : un bouton **« Choisir les fichiers »** apparaît, sélectionnez
`RoadAnomaly.zip`. Le fichier est décompressé dans `/content/data`. Si vous réexécutez la cellule
pendant la même session, le dataset déjà présent est réutilisé (pas de nouvel envoi).

> Les fichiers envoyés sur Colab sont effacés à la fin de la session. Les résultats (figures,
> tableaux) sont téléchargés en ZIP par la dernière cellule du notebook.

In [ ]:
import os, sys, zipfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

def trouver_dataset(dossier):
    # renvoie le dossier qui contient frame_list.json (où qu'il soit dans le zip)
    dossier = Path(dossier)
    if not dossier.exists():
        return None
    hits = sorted(dossier.rglob("frame_list.json"))
    return hits[0].parent if hits else None

if IN_COLAB:
    BASE = Path("/content")
    DATA_ROOT = trouver_dataset(BASE / "data")
    if DATA_ROOT is None:
        from google.colab import files
        print("Sélectionnez RoadAnomaly.zip :")
        envoyes = files.upload()
        for nom in envoyes:
            with zipfile.ZipFile(BASE / nom) as z:
                z.extractall(BASE / "data")
            os.remove(BASE / nom)
        DATA_ROOT = trouver_dataset(BASE / "data")
else:
    # exécution sur votre ordinateur : le notebook doit être à côté du dossier RoadAnomaly
    BASE = Path.cwd()
    DATA_ROOT = next((p for p in [Path("RoadAnomaly"), Path("../RoadAnomaly")]
                      if (p / "frame_list.json").exists()), None)

assert DATA_ROOT is not None, "Dataset introuvable : vérifiez que le zip contient frame_list.json et frames/"
FIG_DIR = BASE / "outputs" / "figures"; FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR = BASE / "outputs" / "tables";  TAB_DIR.mkdir(parents=True, exist_ok=True)
print("Dataset :", DATA_ROOT)
print("Images  :", len(list((DATA_ROOT / "frames").glob("*.webp"))))

### 0.1 Bibliothèques et fonctions utilitaires

Fonctions de chargement réutilisées dans tout le notebook. Le dataset fournit pour chaque image
un dossier `<nom>.labels/` ; dans `labels_semantic.png`, **0 = fond** et **2 = anomalie**.

In [ ]:
import json, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
from PIL import Image
from sklearn.metrics import roc_auc_score

ANOMALY_LABEL = 2
CATEGORIES = ["animals", "obstacles", "cones", "lost_tires", "vehicle"]

# Images provenant de la même source (même vidéo, même photographe, même page web ou même lieu),
# d'après credits.txt. Elles se ressemblent : un groupe ne doit jamais être réparti entre
# entraînement et test (fuite d'information). Utilisé à l'étape 2 pour le découpage.
SOURCE_GROUPS = {
    "youtube_lost_tires": ["lost_tires03", "lost_tires08", "lost_tires09"],
    "cardboard_hedgehogs": ["obstacles03_cardboard_czech_hedgehog", "obstacles04_cardboard_czech_hedgehog",
                            "obstacles05_cardboard_czech_hedgehog", "obstacles06_cardboard_czech_hedgehog"],
    "tripadvisor_rocks": ["obstacles12_rocks2", "obstacles12_rocks4", "obstacles12_rocks5", "obstacles12_rocks6"],
    "papakura_news": ["cones07", "cones08_can"],
    "broadmoor": ["cones02_Broadmoor_Manhole_Cones", "vehicle02_Broadmoor_Cones_Skidloader"],
}
_GROUPE = {n: g for g, noms in SOURCE_GROUPS.items() for n in noms}

def list_frames(root):
    with open(Path(root) / "frame_list.json") as f:
        return [Path(x).stem for x in json.load(f)]

def category_of(name):           # 'cones03_Bang_Na' -> 'cones'
    m = re.match(r"([a-z_]+?)\d", name)
    return m.group(1) if m else name.split("_")[0]

def source_group(name):
    return _GROUPE.get(name, name)

def label_dir(root, name):
    return Path(root) / "frames" / f"{name}.labels"

def load_image(root, name):      # RGB uint8 (H, W, 3)
    return np.array(Image.open(Path(root) / "frames" / f"{name}.webp").convert("RGB"))

def load_semantic(root, name):   # valeurs 0 / 2
    return np.array(Image.open(label_dir(root, name) / "labels_semantic.png"))

def load_mask(root, name):       # masque binaire 0 / 1
    return (load_semantic(root, name) == ANOMALY_LABEL).astype(np.uint8)

def load_instances(root, name):  # identifiant d'instance par pixel, ou None si absent
    p = label_dir(root, name) / "labels_instance.png"
    return np.array(Image.open(p)) if p.exists() else None

def load_boxes(root, name):      # boîtes (x0, y0, x1, y1) depuis index.json
    p = label_dir(root, name) / "index.json"
    if not p.exists():
        return []
    return [(*i["roi_rect"][0], *i["roi_rect"][1]) for i in json.load(open(p)).get("instances", [])]

def overlay(image, mask, color=(0, 200, 255), alpha=0.55):
    out = image.astype(np.float32).copy()
    m = mask.astype(bool)
    out[m] = (1 - alpha) * out[m] + alpha * np.array(color, np.float32)
    return out.clip(0, 255).astype(np.uint8)

# ---- style des graphiques ----
plt.rcParams.update({
    "figure.dpi": 80, "savefig.dpi": 150, "savefig.bbox": "tight",
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#8a8984", "xtick.color": "#52514e", "ytick.color": "#52514e",
    "axes.grid": True, "grid.color": "#e4e3df", "grid.linewidth": 0.8, "axes.axisbelow": True,
})
CAT_COLORS = {"animals": "#2a78d6", "obstacles": "#eb6834", "cones": "#1baf7a",
              "lost_tires": "#eda100", "vehicle": "#e87ba4"}
ANOMALY_RGB = (0, 200, 255)     # cyan : anomalies sur les images (contraste avec les cônes orange)
BLUE, ORANGE, GREY = "#2a78d6", "#eb6834", "#8a8984"

def savefig(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png")

print("OK")

## 1. Structure du dataset et annotations disponibles

| Fichier | Contenu |
|---|---|
| `frames/<nom>.webp` | image RGB |
| `labels_semantic.png` | étiquette par pixel : **0 = fond, 2 = anomalie** |
| `labels_semantic_color.png` | rendu couleur du masque (visualisation uniquement) |
| `labels_instance.png` | un identifiant par objet (0 = fond) |
| `index.json` | une boîte englobante par objet (`roi_rect`) |
| `instance_XXX_gc_mask.png`, `instance_XXX_gc_state.npy` | fichiers internes de l'outil d'annotation LabelGrab |

On vérifie combien d'images disposent de chaque type d'annotation utile.

In [ ]:
frames = list_frames(DATA_ROOT)
print(f"Images listées dans frame_list.json : {len(frames)}")
print(f"Fichiers .webp présents             : {len(list((DATA_ROOT / 'frames').glob('*.webp')))}\n")

fichiers = ["labels_semantic.png", "labels_instance.png", "index.json"]
dispo = pd.DataFrame({f: [(label_dir(DATA_ROOT, n) / f).exists() for n in frames] for f in fichiers},
                     index=frames)
display(pd.DataFrame({"images avec le fichier": dispo.sum(), "manquant": (~dispo).sum()}))

sans_inst = dispo.index[~dispo["labels_instance.png"]].tolist()
print(f"Images sans masque d'instance ({len(sans_inst)}) :")
for n in sans_inst: print("   -", n)

**Ce qu'on observe**

- Le dataset contient **60 images RGB**, chacune avec un **masque sémantique**. C'est la vérité
  terrain de tout le projet.
- **Les masques d'instance et les boîtes (`index.json`) manquent pour 14 images.** Le readme du
  dataset l'explique : certaines images ont été annotées à une autre résolution puis
  redimensionnées. Conséquences :
  - les instances **ne peuvent pas servir de cible d'apprentissage** ;
  - pour les statistiques par objet (§7), on utilise les **composantes connexes du masque
    sémantique**, calculables pour les 60 images.

## 2. Contrôles d'intégrité et métadonnées par image

En un seul passage sur le dataset :

- on vérifie que chaque image et son masque ont la même taille, et que les masques ne contiennent
  que les valeurs documentées (0 et 2) ;
- on rassemble les statistiques de chaque image dans un tableau (`meta`), réutilisé dans la suite.

Un **objet** est une région connexe de pixels anomaux (connexité 8) d'au moins `MIN_OBJ_AREA`
pixels. Les fragments plus petits sont des « miettes » d'annotation (voir §7).

In [ ]:
MIN_OBJ_AREA = 100      # pixels, à pleine résolution

lignes, objets = [], []
for n in frames:
    img  = load_image(DATA_ROOT, n)
    sem  = load_semantic(DATA_ROOT, n)
    mask = (sem == ANOMALY_LABEL).astype(np.uint8)
    inst = load_instances(DATA_ROOT, n)

    assert img.shape[:2] == sem.shape, f"taille différente : {n}"
    assert set(np.unique(sem)) <= {0, ANOMALY_LABEL}, f"valeurs inattendues : {n}"

    k, cc, stats, cent = cv2.connectedComponentsWithStats(mask, connectivity=8)
    aires = stats[1:, cv2.CC_STAT_AREA]
    for i in range(1, k):
        x, y, w, h, a = stats[i]
        objets.append(dict(name=n, category=category_of(n), area=a, w=w, h=h,
                           cx=cent[i][0] / img.shape[1], cy=cent[i][1] / img.shape[0],
                           is_speck=a < MIN_OBJ_AREA))

    gris = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    lignes.append(dict(
        name=n, category=category_of(n), source_group=source_group(n),
        height=img.shape[0], width=img.shape[1], anomaly_frac=mask.mean(),
        n_components=k - 1, n_objects=int((aires >= MIN_OBJ_AREA).sum()),
        n_instances=(len(np.unique(inst)) - 1) if inst is not None else np.nan,
        brightness=gris.mean(), contrast=gris.std()))

meta = pd.DataFrame(lignes)
objs = pd.DataFrame(objets)

print("Tous les contrôles sont passés.")
print("Tailles d'image :", meta.groupby(["width", "height"]).size().to_dict())
print("Valeurs des masques : {0, 2} uniquement")
meta.head()

In [ ]:
meta[["anomaly_frac", "n_components", "n_objects", "n_instances", "brightness", "contrast"]].describe().round(3)

**Les données sont propres.** Les 60 images font **1280 × 720**, chaque masque a la même taille
que son image et ne contient que 0 et 2. Seule transformation nécessaire au prétraitement :
convertir le masque en {0, 1} (`masque = labels_semantic == 2`).

## 3. Répartition des catégories

Le nom des fichiers indique le type d'anomalie (`animals…`, `cones…`, `obstacles…`,
`lost_tires…`, `vehicle…`). Le dataset ne fournit pas d'étiquette de catégorie : on s'en sert
uniquement pour l'analyse et pour stratifier le découpage. **La tâche reste binaire** :
anomalie / non-anomalie.

In [ ]:
cat_counts = meta["category"].value_counts().reindex(CATEGORIES)

fig, ax = plt.subplots(figsize=(7, 3))
bars = ax.barh(cat_counts.index[::-1], cat_counts.values[::-1], height=0.6,
               color=[CAT_COLORS[c] for c in cat_counts.index[::-1]])
for b, v in zip(bars, cat_counts.values[::-1]):
    ax.text(v + 0.4, b.get_y() + b.get_height() / 2, f"{v}  ({v/len(meta):.0%})", va="center", fontsize=9)
ax.set_xlabel("Nombre d'images"); ax.set_xlim(0, cat_counts.max() * 1.25)
ax.grid(axis="y", visible=False)
ax.set_title("Images par catégorie d'anomalie (60 au total)")
savefig(fig, "01_repartition_categories"); plt.show()

**Observations**

- **Les animaux représentent la moitié du dataset** (31/60), puis les obstacles (rochers,
  blocs, hérissons tchèques ; 15) et les cônes (8).
- **Les pneus perdus et les véhicules n'ont que 3 images chacun.** Avec un découpage 70/15/15,
  une telle catégorie aurait au mieux une image de test : un score par catégorie reposerait sur
  une seule image.

**Conséquences**

- Le découpage doit être **stratifié par catégorie**, sinon une catégorie rare peut disparaître
  de la validation ou du test.
- Le jeu de test étant très petit, la **validation croisée en k plis** (ex. 5 plis stratifiés)
  donnera des résultats plus fiables qu'un découpage unique. On rapportera moyenne ± écart-type.

## 4. Inspection visuelle

### 4.1 Un exemple par catégorie
Colonnes : image originale, masque binaire, masque superposé à l'image (en cyan).

In [ ]:
exemples = {"animals": "animals22_red_fox_on_road", "obstacles": "obstacles09_boulder3",
            "cones": "cones03_Bang_Na_Trat_Highway", "lost_tires": "lost_tires03",
            "vehicle": "vehicle01_man_carrying_dry_grass_on_bicycle"}

fig, axes = plt.subplots(len(exemples), 3, figsize=(13, 2.55 * len(exemples)))
for r, (cat, n) in enumerate(exemples.items()):
    img, m = load_image(DATA_ROOT, n), load_mask(DATA_ROOT, n)
    for c, (panel, titre) in enumerate([(img, "Image"), (m, "Masque"),
                                        (overlay(img, m, ANOMALY_RGB), "Superposition")]):
        ax = axes[r, c]
        ax.imshow(panel, cmap="gray" if panel.ndim == 2 else None)
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        if r == 0: ax.set_title(titre)
    axes[r, 0].set_ylabel(f"{cat}\n{m.mean():.1%} anomalie", fontsize=10)
plt.tight_layout(); savefig(fig, "02_exemples_par_categorie"); plt.show()

### 4.2 Vue d'ensemble des 60 images
Anomalies entourées en cyan, images regroupées par catégorie (couleur du titre = catégorie).

In [ ]:
ordre = meta.assign(_c=meta.category.map(CATEGORIES.index)).sort_values(["_c", "name"])
ncols = 6
nrows = int(np.ceil(len(ordre) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 1.55))
for ax in axes.ravel(): ax.axis("off")
for ax, (_, row) in zip(axes.ravel(), ordre.iterrows()):
    img = cv2.resize(load_image(DATA_ROOT, row["name"]), (320, 180), interpolation=cv2.INTER_AREA)
    m = cv2.resize(load_mask(DATA_ROOT, row["name"]), (320, 180), interpolation=cv2.INTER_NEAREST)
    contours, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    ax.imshow(cv2.drawContours(img.copy(), contours, -1, ANOMALY_RGB, 2))
    ax.set_title(row["name"][:24], fontsize=6.5, color=CAT_COLORS[row["category"]], fontweight="bold")
plt.tight_layout(pad=0.3); savefig(fig, "03_galerie_dataset"); plt.show()

**Observations**

- **Les scènes sont très variées.** Les images viennent de Wikimedia, Flickr, de sites
  d'actualité et de vidéos YouTube de caméras embarquées. Pays variés (Laos, Éthiopie, Norvège,
  Maroc…), milieux urbains et ruraux, routes goudronnées et pistes, points de vue très différents
  (hauteur conducteur, piéton, ras du sol pour `obstacles08`). Certaines images sont des captures
  vidéo compressées avec filigrane (`lost_tires`). On est loin d'un dataset homogène comme
  Cityscapes.
- **« Anomalie » a un sens précis : un objet hors des classes habituelles de la conduite.** Les
  personnes, voitures, camions, motos et tramways **ne sont pas** annotés, même sur la route (les
  bergers de `animals24`, les randonneurs de `animals26`, les policiers de `cones03`, le tramway
  de `animals15`). Les animaux, rochers, cônes, pneus et véhicules inhabituels (vélo caché sous
  le foin, chargeuse, charrettes) **le sont**. Cela suit la liste des classes de Cityscapes : ce
  qu'un modèle de scène routière standard connaît est « normal ». Le modèle doit donc distinguer
  une personne (normale) d'un cheval (anomalie) : il faut une compréhension sémantique, pas
  seulement « un objet sur la route ».
- **Les anomalies ne sont pas toujours sur la chaussée** : le bétail au bord de la route
  (`animals05`, `animals10`) est annoté aussi.
- **Les masques sont précis** (pattes du renard, bords des cônes) : la qualité des contours
  comptera pour l'IoU et le Dice.
- Certaines images se ressemblent beaucoup (pneus, hérissons en carton, rochers) : voir §4.3.

### 4.3 Images issues de la même source

Les crédits (`credits.txt`) confirment que plusieurs images partagent la même source (une vidéo
YouTube, la page d'un artiste, un photographe, un article de presse), et deux images le même lieu
(*Broadmoor*). Ces groupes sont définis dans `SOURCE_GROUPS` (cellule 0.1).

In [ ]:
print(f"{len(SOURCE_GROUPS)} groupes couvrant {sum(len(v) for v in SOURCE_GROUPS.values())} images "
      f"-> {meta.source_group.nunique()} sources indépendantes au total\n")

fig, axes = plt.subplots(len(SOURCE_GROUPS), 4, figsize=(13, 1.95 * len(SOURCE_GROUPS)))
for r, (g, noms) in enumerate(SOURCE_GROUPS.items()):
    for c in range(4):
        ax = axes[r, c]; ax.axis("off")
        if c < len(noms):
            img = cv2.resize(load_image(DATA_ROOT, noms[c]), (320, 180), interpolation=cv2.INTER_AREA)
            ax.imshow(img); ax.set_title(noms[c][:30], fontsize=7.5)
    axes[r, 0].text(-0.04, 0.5, g, transform=axes[r, 0].transAxes, rotation=90,
                    ha="right", va="center", fontsize=9, fontweight="bold")
plt.tight_layout(); savefig(fig, "04_groupes_meme_source"); plt.show()

**Pourquoi c'est important : la fuite de données.** Si `lost_tires08` est dans l'entraînement et
`lost_tires09`, presque identique, dans le test, le modèle est évalué sur une scène déjà vue et le
score est surestimé. Le découpage doit donc être **par groupe** : toutes les images d'une source
vont dans le même pli (`StratifiedGroupKFold` de scikit-learn).

Conséquence : **les trois images de pneus viennent d'une seule vidéo** et forment un seul groupe.
Quand ce groupe est en test, le modèle n'a jamais vu de pneu à l'entraînement. C'est un vrai test
de généralisation à un type d'anomalie inconnu, ce qui est justement l'objectif de la détection
d'anomalies. On le soulignera lors de l'analyse par catégorie.

## 5. Formats d'annotation : masque sémantique vs instances

Trois vues de la même annotation :
- le **masque sémantique** (anomalie ou non, par pixel) : la cible de la segmentation ;
- le **masque d'instance** : un identifiant par objet ;
- les **boîtes** de `index.json` (`roi_rect`, coordonnées `[x0, y0], [x1, y1]`).

Comparaison sur une scène chargée (`cones01`) et une scène où les objets se touchent (`animals05`).

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 5.2))
for r, n in enumerate(["cones01_Beeston_MMB_A6_Middle_Street", "animals05_cows_near_Phonsavan_Laos"]):
    img, m, inst = load_image(DATA_ROOT, n), load_mask(DATA_ROOT, n), load_instances(DATA_ROOT, n)
    k, _ = cv2.connectedComponents(m, connectivity=8)
    axes[r, 0].imshow(m, cmap="gray")
    axes[r, 0].set_title(f"Masque sémantique ({k-1} composantes connexes)")
    axes[r, 1].imshow(img // 3 + 120)
    axes[r, 1].imshow(np.ma.masked_where(inst == 0, inst), cmap="tab20", interpolation="nearest")
    axes[r, 1].set_title(f"Masque d'instance ({len(np.unique(inst)) - 1} instances)")
    axes[r, 2].imshow(img)
    for (x0, y0, x1, y1) in load_boxes(DATA_ROOT, n):
        axes[r, 2].add_patch(mpatches.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec=ORANGE, lw=1.5))
    axes[r, 2].set_title("Boîtes englobantes (index.json)")
    for ax in axes[r]: ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    axes[r, 0].set_ylabel(n.split("_")[0], fontsize=10)
plt.tight_layout(); savefig(fig, "05_formats_annotation"); plt.show()

In [ ]:
cmp = meta.dropna(subset=["n_instances"])
diff = cmp["n_objects"] - cmp["n_instances"]
print(f"Images avec masque d'instance : {len(cmp)}")
print(f"  même nombre que les composantes connexes : {(diff == 0).sum()}")
print(f"  moins de composantes que d'instances     : {(diff < 0).sum()}  (objets qui se touchent fusionnés)")
print(f"  plus de composantes que d'instances      : {(diff > 0).sum()}  (objet coupé en morceaux)")

**À retenir**

- Les composantes connexes sont une **bonne approximation, mais imparfaite**, des objets : un
  troupeau de vaches qui se touchent **fusionne** en une composante ; un objet fin ou
  partiellement caché peut **se scinder** en plusieurs.
- Ce n'est pas gênant : on modélise la tâche en **segmentation sémantique binaire**, et le nombre
  et la taille des objets servent seulement à décrire les données et à analyser les erreurs par
  taille d'objet.
- Les boîtes permettraient une approche par détection (ex. Faster R-CNN). On ne la retient pas :
  les boîtes manquent pour 14 images, et le sujet demande des métriques pixel (IoU, Dice).

## 6. Déséquilibre des classes au niveau pixel

En segmentation, chaque pixel est un exemple d'apprentissage : ce qui compte est la **proportion
de pixels** de chaque classe, pas le nombre d'images.

In [ ]:
total_px   = (meta["height"] * meta["width"]).sum()
anomaly_px = (meta["anomaly_frac"] * meta["height"] * meta["width"]).sum()
p = anomaly_px / total_px
print(f"Pixels anomaux sur l'ensemble du dataset : {p:.2%}")
print(f"Rapport fond : anomalie                  : {(1-p)/p:.1f} : 1")
print(f"Par image -> médiane {meta.anomaly_frac.median():.2%}, "
      f"min {meta.anomaly_frac.min():.2%}, max {meta.anomaly_frac.max():.2%}")
print(f"Images avec moins de 2 % de pixels anomaux : {(meta.anomaly_frac < 0.02).sum()} / {len(meta)}")
print(f"\nUn modèle qui prédit « fond » partout obtiendrait {1-p:.1%} d'exactitude pixel,"
      f" mais une IoU de 0 sur la classe anomalie.")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 3.8), gridspec_kw={"width_ratios": [1, 1.25]})

pct = meta["anomaly_frac"] * 100
ax1.hist(pct, bins=np.arange(0, 46, 2.5), color=BLUE, edgecolor="white", linewidth=1.5)
ax1.axvline(pct.median(), color="#2b2b2b", lw=1.2, ls="--")
ax1.text(pct.median() + 0.8, ax1.get_ylim()[1] * 0.92, f"médiane {pct.median():.1f} %", fontsize=9)
ax1.set_xlabel("Pixels anomaux dans l'image (%)"); ax1.set_ylabel("Nombre d'images")
ax1.set_title("(a) Part de pixels anomaux par image"); ax1.grid(axis="x", visible=False)

rng = np.random.default_rng(0)
for i, c in enumerate(CATEGORIES):
    v = meta.loc[meta.category == c, "anomaly_frac"] * 100
    ax2.plot([v.median()] * 2, [i - 0.3, i + 0.3], color="#2b2b2b", lw=1.5, zorder=2)
    ax2.scatter(v, i + rng.uniform(-0.15, 0.15, len(v)), s=36, color=CAT_COLORS[c],
                edgecolor="white", linewidth=1, zorder=3)
ax2.set_yticks(range(len(CATEGORIES))); ax2.set_yticklabels(CATEGORIES)
ax2.invert_yaxis(); ax2.grid(axis="y", visible=False)
ax2.set_xlabel("Pixels anomaux dans l'image (%)")
ax2.set_title("(b) Par catégorie (points = images, trait = médiane)")
plt.tight_layout(); savefig(fig, "06_desequilibre_pixels"); plt.show()

**Observations**

- Globalement **9,9 % des pixels sont anomaux**, soit un rapport fond:anomalie d'environ
  **9 : 1**. La moyenne masque une distribution très asymétrique : **l'image médiane n'a que
  6,6 % de pixels anomaux**, et **16 images sur 60 en ont moins de 2 %**. Quelques gros plans
  (moutons, bœufs, ânes : 30 à 42 %) tirent la moyenne vers le haut.
- **Le déséquilibre dépend de la catégorie.** Cônes et pneus sont presque toujours sous 2 %
  (petits objets lointains). Animaux et obstacles vont de moins de 1 % à plus de 40 %.

**Conséquences pour la modélisation**

- **L'exactitude pixel n'a aucun sens ici.** On utilisera précision, rappel, F1, IoU et Dice sur
  la classe anomalie, plus des métriques indépendantes du seuil (AUROC, AUPRC / Average
  Precision, FPR à 95 % de TPR), standard sur ce benchmark.
- **L'entropie croisée binaire seule favorise le fond.** On entraînera avec une perte adaptée au
  déséquilibre : **Dice loss** (optimise directement le recouvrement) combinée à **BCE ou Focal
  loss** (gradients stables, calibration pixel). Un poids de classe positive proche du rapport
  fond:anomalie est une alternative à tester.
- Comme la couverture varie beaucoup, on calculera l'**IoU sur l'ensemble des pixels du test**
  (niveau dataset) et aussi la **moyenne par image** (qui donne plus de poids aux petites
  anomalies).

## 7. Analyse au niveau objet : nombre et taille

Les réseaux de segmentation ont surtout du mal avec les **petits objets** : après plusieurs
sous-échantillonnages, un cône de 20 pixels ne couvre plus qu'une ou deux cellules de la carte de
caractéristiques. On étudie donc la taille de chaque objet anomal.

Classes de taille selon la convention COCO : **petit** < 32² px, **moyen** 32²–96² px,
**grand** > 96² px (aire mesurée à la résolution 1280 × 720).

In [ ]:
miettes = objs[objs.is_speck]
reels   = objs[~objs.is_speck].copy()
print(f"Composantes connexes au total : {len(objs)}")
print(f"  miettes (< {MIN_OBJ_AREA} px)          : {len(miettes)} -> "
      f"{miettes.area.sum() / objs.area.sum():.2%} des pixels anomaux (bruit d'annotation)")
print(f"  objets (>= {MIN_OBJ_AREA} px)          : {len(reels)}")

reels["taille"] = pd.cut(reels.area, [0, 32**2, 96**2, np.inf],
                         labels=["petit (<32²)", "moyen (32²–96²)", "grand (>96²)"])
size_tab = pd.crosstab(reels.category, reels.taille).reindex(CATEGORIES)
size_tab["total"] = size_tab.sum(axis=1)
display(size_tab)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 3.8))

cnt = meta["n_objects"].value_counts().sort_index()
ax1.bar(cnt.index, cnt.values, width=0.75, color=BLUE)
ax1.set_xlabel("Nombre d'objets dans l'image"); ax1.set_ylabel("Nombre d'images")
ax1.set_title("(a) Objets par image"); ax1.grid(axis="x", visible=False)
ax1.xaxis.set_major_locator(plt.MaxNLocator(integer=True))

bins = np.logspace(2, np.log10(reels.area.max() * 1.05), 28)
ax2.hist(reels.area, bins=bins, color=BLUE, edgecolor="white", linewidth=1.2)
ax2.set_xscale("log")
for seuil, lab in [(32**2, "32²"), (96**2, "96²")]:
    ax2.axvline(seuil, color="#2b2b2b", ls="--", lw=1)
    ax2.text(seuil * 1.08, ax2.get_ylim()[1] * 0.9, lab, fontsize=9)
ax2.set_xlabel("Aire de l'objet (pixels, échelle log, à 1280×720)"); ax2.set_ylabel("Nombre d'objets")
ax2.set_title("(b) Taille des objets"); ax2.grid(axis="x", visible=False)
plt.tight_layout(); savefig(fig, "07_nombre_et_taille_objets"); plt.show()

### 7.1 Effet de la réduction de résolution sur les petits objets

Entraîner à 1280 × 720 coûte cher sur un GPU gratuit : on réduira probablement la taille des
images. Le tableau montre combien d'objets deviennent **minuscules** (< 8 × 8 = 64 px, environ
une cellule d'une carte de caractéristiques de pas 8) selon la résolution d'entrée.

In [ ]:
res = []
for (W, H) in [(1280, 720), (960, 540), (640, 360), (512, 288), (384, 216)]:
    a = reels.area * (W / 1280) * (H / 720)
    res.append({"résolution": f"{W}×{H}", "aire médiane d'un objet (px)": int(np.median(a)),
                "objets < 64 px": int((a < 64).sum()), "% objets < 64 px": f"{(a < 64).mean():.0%}",
                "images concernées": reels.loc[a < 64, "name"].nunique()})
res_tab = pd.DataFrame(res)
display(res_tab)

**Observations**

- 298 composantes connexes, dont **105 miettes de moins de 100 px qui ne représentent que 0,05 %
  des pixels anomaux** : de petits artefacts d'annotation, sans effet sur l'apprentissage, mais à
  exclure du comptage d'objets.
- Les **193 objets restants couvrent plus de trois ordres de grandeur**, d'environ 100 px (un
  rocher lointain) à plus de 300 000 px (un troupeau de moutons en gros plan).
- **La plupart des images contiennent 1 à 3 objets** (28 n'en ont qu'un), mais les scènes
  encombrées (`cones01`, rochers éparpillés) en contiennent 11 à 19.
- **La taille dépend de la catégorie.** Les animaux sont surtout grands (40 objets sur 52
  au-dessus de 96² px). Les petits objets sont concentrés dans les **obstacles (47 petits :
  rochers éparpillés)** et les **cônes (22 petits)**.
- **Réduire la résolution coûte cher.** À 960 × 540 presque tous les objets restent au-dessus de
  64 px. À 640 × 360, 18 % passent en dessous, et 21 % à 512 × 288 (dans 11 images). Ce sont
  justement les cônes, rochers et pneus qu'un véhicule ne doit pas manquer.

**Conséquences**

- La résolution d'entrée est un **compromis entre vitesse et rappel sur les petits objets**. Plan
  raisonnable : **entraîner sur des découpes aléatoires (ex. 512 × 512) d'images à échelle
  modérée**, et **prédire sur l'image entière**. Les réseaux de segmentation étant entièrement
  convolutifs, ils acceptent une image plus grande au test.
- On privilégie les architectures qui conservent les détails fins : **U-Net** (connexions de
  saut depuis les premières couches) et **DeepLabV3+** (décodeur avec caractéristiques bas niveau
  et convolutions dilatées qui limitent le sous-échantillonnage).
- À l'évaluation, on **détaillera les résultats par taille d'objet** (petit / moyen / grand).

## 8. Répartition spatiale des anomalies

Où les anomalies apparaissent-elles dans l'image ? La carte ci-dessous est la **moyenne des 60
masques** : chaque pixel indique la proportion d'images où il est anomal.

In [ ]:
H, W = 720, 1280
heat = np.zeros((H, W), np.float64)
for n in frames:
    heat += load_mask(DATA_ROOT, n)
heat /= len(frames)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 3.9), gridspec_kw={"width_ratios": [2.6, 1]})
im = ax1.imshow(heat, cmap="Blues", vmin=0)
ax1.set_title("(a) Fréquence d'anomalie par pixel (moyenne des 60 masques)")
ax1.set_xticks([]); ax1.set_yticks([]); ax1.grid(False)
cb = fig.colorbar(im, ax=ax1, fraction=0.025, pad=0.02)
cb.set_label("proportion d'images"); cb.outline.set_visible(False)

profil = heat.mean(axis=1) * 100
ax2.plot(profil, np.arange(H), color=BLUE, lw=2)
ax2.fill_betweenx(np.arange(H), 0, profil, color=BLUE, alpha=0.15, lw=0)
ax2.set_ylim(H, 0); ax2.set_xlim(left=0)
ax2.set_xlabel("Pixels anomaux moyens dans la ligne (%)"); ax2.set_ylabel("Ligne de l'image (0 = haut)")
ax2.set_title("(b) Profil vertical")
plt.tight_layout(); savefig(fig, "08_carte_spatiale"); plt.show()

haut = int(H * 0.30)
print(f"Part des pixels anomaux dans les 30 % supérieurs de l'image : {heat[:haut].sum() / heat.sum():.1%}")
print(f"Part dans la moitié inférieure                              : {heat[H//2:].sum() / heat.sum():.1%}")

**Observations**

- Les anomalies se concentrent dans une **bande horizontale au milieu de l'image (lignes
  ≈ 300–500 sur 720)**, juste sous l'horizon : dans une vue de conduite typique, c'est là que se
  trouvent la route et ce qui est dessus. Le pic atteint environ 21 % des pixels d'une ligne.
- Les **30 % supérieurs de l'image ne contiennent que 8 % des pixels anomaux**, la moitié
  inférieure 67 %. Les toutes dernières lignes (près de la caméra) sont aussi rarement anomales.
- Horizontalement, les anomalies sont réparties sur toute la largeur, un peu plus à gauche du
  centre.

**Conséquences**

- Il existe un fort **a priori de position** : les anomalies sont sur la route ou à côté, rarement
  en haut de l'image (ciel, cimes, façades). Les découpes d'entraînement ne doivent pas être
  gaspillées sur cette zone ; centrer une partie des découpes (ex. 50 %) sur des pixels anomaux
  aide aussi contre le déséquilibre.
- Revers de la médaille : un **risque d'apprentissage par raccourci**. Le modèle pourrait
  apprendre « la bande sous l'horizon = anomalie » au lieu de l'aspect des objets. Les
  **retournements horizontaux et les changements d'échelle/découpes aléatoires** limitent ce
  risque ; on vérifiera à l'analyse des erreurs si les faux positifs se concentrent dans cette
  zone.

## 9. Apparence : la couleur suffit-elle à détecter les anomalies ?

Si les anomalies étaient toujours de couleur vive (comme les cônes orange), un simple seuil de
couleur suffirait et le deep learning serait inutile. On compare les **statistiques de couleur des
pixels anomaux et des pixels de fond** (espace HSV, échantillon aléatoire de pixels).

In [ ]:
rng = np.random.default_rng(42)
ech = {"anomalie": [], "fond": []}
for n in frames:
    hsv = cv2.cvtColor(load_image(DATA_ROOT, n), cv2.COLOR_RGB2HSV).reshape(-1, 3)
    m = load_mask(DATA_ROOT, n).ravel().astype(bool)
    for cle, sel in [("anomalie", m), ("fond", ~m)]:
        idx = np.flatnonzero(sel)
        ech[cle].append(hsv[rng.choice(idx, size=min(3000, len(idx)), replace=False)])
ech = {k: np.concatenate(v) for k, v in ech.items()}

fig, axes = plt.subplots(1, 3, figsize=(14, 3.4))
noms   = ["Teinte (OpenCV 0–179)", "Saturation (0–255)", "Valeur / luminosité (0–255)"]
plages = [(0, 180), (0, 256), (0, 256)]
for ch, ax in enumerate(axes):
    for cle, col in [("fond", GREY), ("anomalie", ORANGE)]:
        ax.hist(ech[cle][:, ch], bins=48, range=plages[ch], density=True,
                histtype="step", lw=2, color=col, label=cle)
    ax.set_title(noms[ch]); ax.set_yticks([]); ax.grid(axis="x", visible=False)
axes[0].legend(frameon=False)
plt.tight_layout(); savefig(fig, "09_distributions_couleur"); plt.show()

y = np.r_[np.ones(len(ech["anomalie"])), np.zeros(len(ech["fond"]))]
for ch, nm in enumerate(["Teinte", "Saturation", "Valeur"]):
    x = np.r_[ech["anomalie"][:, ch], ech["fond"][:, ch]].astype(float)
    auc = roc_auc_score(y, x)
    print(f"AUROC pixel avec la {nm:<10}: {max(auc, 1 - auc):.3f}   (0,5 = hasard, 1,0 = parfait)")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.scatter(meta.brightness, meta.contrast, s=40, color=BLUE, edgecolor="white", linewidth=1, zorder=3)
extremes = pd.concat([meta.nsmallest(1, "brightness"), meta.nlargest(1, "brightness"),
                      meta.nsmallest(1, "contrast"), meta.nlargest(1, "contrast")]).drop_duplicates("name")
for _, r in extremes.iterrows():
    ax.annotate(r["name"][:22], (r.brightness, r.contrast), xytext=(6, 4),
                textcoords="offset points", fontsize=7.5, color="#52514e")
ax.set_xlabel("Luminosité moyenne (niveau de gris)"); ax.set_ylabel("Contraste (écart-type des gris)")
ax.set_title("Luminosité et contraste par image (un point = une image)")
plt.tight_layout(); savefig(fig, "10_luminosite_contraste"); plt.show()

**Observations**

- **Teinte :** les pixels anomaux culminent sur les teintes orange/brun (≈ 10–25 : pelage,
  cônes, rochers). Le fond a son propre pic sur le bleu (≈ 105), surtout le ciel. Le pic brun est
  **partagé** avec le fond (pistes, terre, bâtiments).
- **Luminosité / saturation :** les anomalies sont en moyenne un peu plus sombres et plus
  saturées, mais les distributions se recouvrent largement.
- **Un seul canal de couleur utilisé comme classifieur pixel donne une AUROC de 0,60 à 0,63
  seulement** (0,5 = hasard). Ce faible signal vient en bonne partie du ciel, que l'a priori de
  position exclut déjà.
- **Au niveau image :** la luminosité va de 60 à 163 et le contraste de 30 à 84. L'exposition et
  la qualité des caméras varient beaucoup (`obstacles02_rock` très sombre, captures de caméra
  embarquée des pneus très contrastées). D'où l'intérêt d'une **augmentation
  luminosité/contraste** et d'une normalisation par canal (statistiques ImageNet pour les
  encodeurs pré-entraînés).

**Pourquoi c'est important pour le choix des modèles**

Les statistiques de couleur bas niveau séparent à peine les anomalies de la route. Ce qui fait
qu'un objet est une anomalie, c'est **ce qu'il est et où il est** (un animal ou un rocher sur la
chaussée) : une information sémantique et contextuelle. C'est exactement ce qu'apportent les
**réseaux convolutifs pré-entraînés sur de grands datasets** : leurs couches profondes encodent
formes et textures d'objets. C'est l'argument principal en faveur du **transfer learning**,
d'autant plus avec seulement 60 images annotées.

## 10. Synthèse et conséquences pour la suite

| # | Constat | Chiffres | Conséquence pour le projet |
|---|---|---|---|
| 1 | Très petit dataset | 60 images, seulement 50 sources indépendantes | **Transfer learning** (encodeurs pré-entraînés ImageNet), **augmentation de données** forte, **validation croisée** plutôt qu'un découpage unique |
| 2 | Catégories déséquilibrées | animaux 52 %, pneus et véhicules 3 images chacun | Découpage **stratifié** par catégorie ; prudence sur les scores des catégories rares |
| 3 | Images de même source | 5 groupes, 15 images (§4.3) | Plis **par groupe** (`StratifiedGroupKFold`) pour éviter la fuite de données |
| 4 | Fort déséquilibre pixel | 9,9 % de pixels anomaux ; 16 images < 2 % | Perte **Dice + BCE/Focal** ; évaluation par **précision, rappel, F1, IoU, Dice, AUPRC, FPR@95**, jamais l'exactitude pixel |
| 5 | Tailles d'objets sur > 3 ordres de grandeur | 21 % des objets < 64 px à 512 × 288 | **Découpes à échelle modérée** à l'entraînement, prédiction sur l'image entière ; **U-Net / DeepLabV3+** ; résultats **par taille d'objet** |
| 6 | Fort a priori de position | 67 % des pixels anomaux dans la moitié basse, 8 % dans les 30 % du haut | Découpes orientées vers les anomalies, retournements, changements d'échelle ; vérifier l'**apprentissage par raccourci** |
| 7 | La couleur seule ne suffit pas | AUROC ≤ 0,63 par canal | Il faut des caractéristiques sémantiques → **CNN pré-entraîné** plutôt que des descripteurs de couleur |
| 8 | « Anomalie » = objet hors des classes Cityscapes | personnes et voitures non annotées ; animaux et débris annotés | Justifie l'**approche hors-distribution** avec un segmenteur entraîné sur Cityscapes ; les modèles supervisés doivent l'apprendre avec peu d'exemples |
| 9 | Instances incomplètes | absentes pour 14 / 60 images | Tâche formulée en **segmentation sémantique binaire** ; composantes connexes pour l'analyse par objet |
| 10 | Miettes d'annotation | 105 composantes < 100 px, 0,05 % des pixels anomaux | Ignorées dans le comptage ; pas de nettoyage des masques nécessaire |

**Étape suivante — `02_pretraitement.ipynb` :** masques binaires → découpage en 5 plis stratifiés
et par groupe → `Dataset` PyTorch avec augmentations (albumentations) → vérification visuelle des
lots augmentés.

In [ ]:
# Sauvegarde des tableaux (réutilisés à l'étape 2) puis téléchargement des résultats
meta.to_csv(TAB_DIR / "metadata.csv", index=False)
objs.to_csv(TAB_DIR / "objects.csv", index=False)
size_tab.to_csv(TAB_DIR / "taille_objets_par_categorie.csv")
res_tab.to_csv(TAB_DIR / "petits_objets_vs_resolution.csv", index=False)
print("Tableaux :", *sorted(p.name for p in TAB_DIR.iterdir()), sep="\n  ")
print("Figures  :", *sorted(p.name for p in FIG_DIR.iterdir()), sep="\n  ")

if IN_COLAB:
    import shutil
    from google.colab import files
    shutil.make_archive(str(BASE / "resultats_etape1"), "zip", BASE / "outputs")
    files.download(str(BASE / "resultats_etape1.zip"))